# A/B Testing – The ShopNow Checkout Experiment 🛒🎲

This notebook complements the lecture slides in `slides/ab_testing/index.html`.

## Story
**ShopNow** built a new one-page checkout (variant **B**). The old three-page checkout is variant **A**.
Today **5.0 %** of checkout visitors buy. The product team believes B lifts this by **at least 0.5 percentage points** (to 5.5 %).

We are the data team. Our job:

1. **Plan** the experiment: how many users do we need?
2. **Simulate** the experiment (we play the role of the website and the customers).
3. **Analyze** it: is the difference real, or just noise?
4. See what goes wrong when you **peek** every day.
5. Analyze a **continuous metric** (revenue per visitor).
6. Run the **sanity checks** every experiment needs.

Why simulate instead of using a real dataset? Because in a simulation **we know the true effect**, so we can check whether the test finds it. That is the best way to build intuition for noise.

## Tools
- `statsmodels` for sample size and the test for proportions
- `scipy.stats` for the t-test and the chi-squared test
- `pandas` and `matplotlib` as usual

In [ ]:
# Colab has these preinstalled. If you run locally: pip install statsmodels scipy pandas matplotlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportion_effectsize,
    proportions_ztest,
    confint_proportions_2indep,
)
from scipy import stats

pd.set_option("display.precision", 4)

## Exercise 1: Plan the sample size 📏

Four inputs, three of them are decisions:

| knob | ShopNow | who decides |
|---|---|---|
| baseline conversion rate | 5.0 % | the data (BI dashboard) |
| minimum detectable effect (MDE) | +0.5 pp (5.0 % → 5.5 %) | business |
| significance level α | 5 % | convention |
| power (1 − β) | 80 % | convention |

In [ ]:
BASELINE = 0.05   # 5.0 % of checkout visitors buy today
MDE = 0.005       # we want to detect at least +0.5 percentage points
ALPHA = 0.05
POWER = 0.80
VISITORS_PER_DAY = 2000   # checkout visitors per day, both groups together

In [ ]:
# Task 1: Compute the required number of users PER GROUP.
# Hint: effect = proportion_effectsize(BASELINE + MDE, BASELINE)
#       n = NormalIndPower().solve_power(effect_size=effect, alpha=ALPHA, power=POWER, alternative="two-sided")

In [ ]:
# Task 2: How many days does the experiment take with VISITORS_PER_DAY visitors (split 50/50)?

In [ ]:
# Task 3: Plot the power curve: power (y) vs users per group (x) for an MDE of +0.5 pp and of +1.0 pp.
# Draw a horizontal line at 80 %.
# Hint: NormalIndPower().power(effect_size=..., nobs1=n, alpha=ALPHA, ratio=1.0) gives the power for n users per group.
ns = np.arange(1_000, 60_000, 1_000)

In [ ]:
# Task 4: The business says: "We can only run this for 14 days."
# What is the smallest effect (in percentage points) we can detect with 80 % power in 14 days?
# Hint: loop over candidate MDEs (e.g. 0.001, 0.002, ..., 0.02), compute the power for the users
#       we get in 14 days, and report the first MDE that reaches 80 %.

## Exercise 2: Simulate the experiment 🎲

We play the website: every visitor gets a coin flip (A or B), then "decides" to buy with the probability of their group.

The function below is almost complete. It also simulates the device of the user and the basket value of buyers, which we need later.

In [ ]:
def simulate_experiment(n_users, p_a, p_b, seed=0, share_b=0.5):
    """Simulate one A/B test.

    n_users : total number of visitors
    p_a, p_b: conversion probability in group A and B
    share_b : probability of being assigned to B (0.5 = fair coin)

    Returns a DataFrame with one row per visitor:
    user_id, variant, device, converted (0/1), revenue (0 if not converted)
    """
    rng = np.random.default_rng(seed)

    # Task 1: assign each user to "A" or "B" with probability share_b for "B".
    # Hint: rng.choice(["A", "B"], size=n_users, p=[1 - share_b, share_b])
    variant = ...

    device = rng.choice(["mobile", "desktop"], size=n_users, p=[0.6, 0.4])

    # Task 2: draw the conversion: 1 with probability p_a for A users and p_b for B users, else 0.
    # Hint: p = np.where(variant == "B", p_b, p_a); converted = (rng.random(n_users) < p).astype(int)
    converted = ...

    # basket value of buyers: skewed, mean about 60 EUR
    basket = rng.lognormal(mean=np.log(50), sigma=0.6, size=n_users)
    revenue = converted * basket

    return pd.DataFrame({
        "user_id": np.arange(n_users),
        "variant": variant,
        "device": device,
        "converted": converted,
        "revenue": revenue.round(2),
    })

In [ ]:
# Task 3: Run the experiment with the planned number of users (2 * n_per_group),
# a true conversion rate of 5.0 % in A and 5.5 % in B. Look at the first rows.

In [ ]:
# Task 4: Summarize per variant: number of users, number of buyers, conversion rate.
# Hint: groupby("variant").agg(...)

## Exercise 3: Analyze the result 🔬

Now the real question: is the difference between A and B bigger than what noise alone would produce?

The tool: a **two-sample z-test for proportions** (`proportions_ztest`) and the **confidence interval** for the difference (`confint_proportions_2indep`).

In [ ]:
# Task 1: Run the z-test. Print the z statistic and the p-value.
# Hint: proportions_ztest(count=[buyers_A, buyers_B], nobs=[users_A, users_B])

In [ ]:
# Task 2: Compute the difference B - A in percentage points and its 95 % confidence interval.
# Hint: confint_proportions_2indep(count1, nobs1, count2, nobs2) returns (low, high) for p1 - p2.
#       Put B first so that the interval is for B - A.

**Task 3: Decide.** Write your decision in 2-3 sentences, as you would report it to the product team: effect, interval, p-value, sample size.

*Your answer here:*

In [ ]:
# Task 4: Did the test keep its promise?
# (a) Simulate 200 A/A tests (p_a = p_b = 0.05, same N_USERS) and count how often p < 0.05. Expectation: about 5 %.
# (b) Simulate 200 A/B tests with the true effect (p_b = 0.055) and count how often p < 0.05. Expectation: about 80 % (the power).
# Hint: write a small helper that simulates and returns the p-value, then loop with different seeds.

## Exercise 4: The peeking problem 👀

In reality the data arrives day by day and everybody has a dashboard. Let's see what happens if we test **every day** and stop as soon as p < 0.05.

To make the point, we simulate an **A/A test**: there is no effect at all.

In [ ]:
def daily_pvalues(n_days, visitors_per_day, p_a, p_b, seed):
    """Return the p-value you would get if you ran the test at the end of each day (cumulative data)."""
    df = simulate_experiment(n_days * visitors_per_day, p_a, p_b, seed=seed)
    df["day"] = np.repeat(np.arange(1, n_days + 1), visitors_per_day)
    pvals = []
    for day in range(1, n_days + 1):
        # Task 1: take all data up to and including `day`, compute buyers/users per variant and the p-value.
        ...
        pvals.append(...)
    return np.array(pvals)

In [ ]:
# Task 2: Plot the daily p-value of 5 A/A tests (30 days, 2000 visitors/day, p_a = p_b = 0.05) in one chart,
# with a horizontal line at 0.05. How many of them dip below the line at some point?

In [ ]:
# Task 3: Repeat for 100 A/A tests. Count:
#  (a) how many are "significant" at the end (day 30)       -> should be about 5 %
#  (b) how many are "significant" on at least one day       -> the false alarm rate of a peeker
# (This takes a minute: 100 tests x 30 days.)

**Task 4: Explain** in your own words why peeking breaks the test, and what you should do instead.

*Your answer here:*

## Exercise 5: A continuous metric: revenue per visitor 💶

Conversion is binary (0/1). Revenue per visitor is continuous: most visitors spend 0, buyers spend a skewed amount.
For means we use the **t-test** (`scipy.stats.ttest_ind`).

In [ ]:
# Task 1: Using `df` from Exercise 2: plot a histogram of the revenue of BUYERS only (revenue > 0), per variant.
# What does the distribution look like?

In [ ]:
# Task 2: Compute the mean revenue PER VISITOR (including the zeros) for A and B,
# then run a t-test (Welch: equal_var=False) and print the p-value and the 95 % confidence interval of the difference.
# Hint: res = stats.ttest_ind(rev_b, rev_a, equal_var=False); res.pvalue; res.confidence_interval()

**Task 3:** Compare with the conversion result from Exercise 3. Is the revenue result as clear? Why (not)?

*Your answer here:*

## Exercise 6: Sanity checks 🩺

Before anyone believes an experiment result, two checks:

1. **Sample ratio mismatch (SRM):** we planned a 50/50 split. Is the observed split compatible with that?
2. **Segments:** does the result hold in every segment, or is something odd going on (Simpson's paradox)?

In [ ]:
# Task 1: SRM check for `df` from Exercise 2.
# Hint: counts = df["variant"].value_counts(); stats.chisquare(counts) tests against an equal split.
# Then do the same for an experiment simulated with share_b=0.49 and 100,000 users. Is that a bug?

In [ ]:
# Task 2: Segment check. Compute the conversion rate per device and variant for `df` (a pivot table: device x variant).
# Is B better on both devices?

In [ ]:
# Task 3 (Simpson's paradox): Build the case from the lecture by hand and check it.
#   mobile:  A 1000 users, 3.0 % conversion;  B 4000 users, 3.5 %
#   desktop: A 4000 users, 8.0 % conversion;  B 1000 users, 9.0 %
# Compute the overall conversion rate of A and B. Who wins overall? Who wins per segment? Why?

## Optional: a real dataset 🌍

If you want to try this on real data, the Kaggle dataset
[Marketing A/B Testing](https://www.kaggle.com/datasets/faviovaz/marketing-ab-testing) contains about 588,000 users
who saw either an ad (`test group = "ad"`) or a public service announcement (`"psa"`), and whether they converted.

1. Download `marketing_AB.csv` from Kaggle (free account) and upload it to Colab (folder icon on the left).
2. Load it with `pd.read_csv("marketing_AB.csv")`.
3. Compute users, buyers and the conversion rate per `test group`.
4. Run the SRM check. (Spoiler: the split is far from 50/50. Is that a bug, or was it planned that way?)
5. Run the z-test and compute the confidence interval of the lift.
6. Is the lift relevant for the business? What else would you want to know before recommending the ad?

## Summary

- **Plan first:** baseline, MDE, α and power give you the sample size. Without it you do not know when to stop.
- **Randomize** per user with a fair coin. Check the split (SRM).
- **One test, at the end.** Peeking turns 5 % false alarms into 25 % or more.
- **Report effect + confidence interval**, not just "significant".
- **Noisy metrics need more users.** Conversion before revenue.
- **Check segments.** A result that only holds overall is suspicious.